# Task3 精修诊断v3：内层选轮次＋裁剪扰动／位置先验消融

固定现有512粗预测，公开参考14.16538。上一版粗精等权公开16.45402，当前最好仍为320全量模型12.31624。本轮只做有控制的修复实验，不保证提升。

三个精修对照：**wide**：原每轴±160均匀扰动；**small**：较小扰动；**small_prior**：与small共用训练网络，在推理热图上加入固定粗定位位置先验。骨干、损失、768→384裁剪、图像增强与256像素回退规则保持一致。

外层每折约64张训练/16张评估。每个64张内部再分约51/13：辅助粗模型仅训练内层训练图，为13张产生真正未见过这些图片的粗坐标。内层选精修轮次；再用外层全部64张从ImageNet重训选定轮数，最后一次评估外层16张。

**评估边界：**新增精修不再用外层标签选轮次/门控。但固定的旧粗模型本身曾用这些外层标签选过轮次，因此这里只能称“给定旧粗模型的开发评估”，不能声称整条级联已有完全独立的嵌套验证。

已内置上轮22个粗预测/元数据文件，只需挂载比赛图片。无需上传大型模型或其他结果ZIP。


## 1. 配置

在Kaggle导入Notebook，添加2026task3比赛数据，开启GPU和Internet。只使用cuda:0，各阶段顺序运行。完整实验最多25个训练阶段：5个内层辅助粗模型、10个内层精修拟合、最多10个外层重训。小扰动的两种解码共用一次训练及对应轮次快照。

按上一轮T4日志粗估约30～60分钟，实际取决于内层选中轮数和门控；完整备份压缩另计。可以先RUN_FOLDS=[0]完成一折，再补其余。SMOKE_TEST=True只做短流程检查，会使用独立目录且禁止导出正式候选。


In [ ]:
from pathlib import Path
if not Path('/kaggle/working').is_dir():
    raise RuntimeError('本Notebook只在Kaggle GPU运行；本机不导入Torch或启动训练。')
DATA_DIR=None
COARSE_SOURCE=None  # 默认使用内置固定粗预测
SMOKE_TEST=False
OUTPUT_DIR=Path('/kaggle/working/task3_refine_diagnostic_v3'+('_smoke' if SMOKE_TEST else ''))
RESUME_DIR=None     # 本版完整Output目录，直接包含本版protocol.json
WEIGHTS=None
RUN_FOLDS=[0] if SMOKE_TEST else [0,1,2,3,4]
CREATE_FULL_BACKUP=True


In [ ]:
from __future__ import annotations
from collections import defaultdict
from contextlib import contextmanager
import base64, csv, gc, hashlib, io, json, math, os, random, shutil, statistics, sys, time, zipfile
import xml.etree.ElementTree as ET
from pathlib import Path, PurePosixPath
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
WEIGHT_NAME='resnet34-b627a593.pth'
WEIGHT_URL='https://download.pytorch.org/models/'+WEIGHT_NAME

def save_json(path, value):
    path = Path(path)
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(4)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def find_data(explicit=None):
    if explicit:
        candidates = [Path(explicit)]
    else:
        base = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
        candidates = [base / '2026task3/detection', Path.cwd() / '2026task3/detection',
                      Path.cwd() / 'task3/2026task3/detection']
        if Path('/kaggle/input').exists():
            candidates += [p.parent for p in Path('/kaggle/input').rglob('fovea_localization_train_GT.csv')]
    valid = sorted({p.resolve() for p in candidates
                    if (p / 'fovea_localization_train_GT.csv').is_file()
                    and (p / 'sample_submission.csv').is_file()
                    and (p / 'train').is_dir() and (p / 'test').is_dir()})
    if len(valid) != 1:
        raise ValueError(f'Expected one dataset; set data_dir explicitly. Found: {valid}')
    return valid[0]


def load_manifest(data):
    labels = pd.read_csv(data / 'fovea_localization_train_GT.csv')
    if list(labels.columns) != ['data', 'Fovea_X', 'Fovea_Y'] or labels.isna().any().any():
        raise ValueError('Invalid training CSV columns or missing values.')
    if labels['data'].duplicated().any():
        raise ValueError('Duplicate training IDs.')
    lookup = labels.set_index('data')
    records = []
    for subset in ['train', 'test']:
        files = sorted(p for p in (data / subset).iterdir()
                       if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
        for p in files:
            image_id = int(p.stem)
            with Image.open(p) as im:
                im = im.convert('RGB')
                im.load()
                w, h = im.size
                # Hash decoded pixels, not JPEG metadata. Size participates in the hash.
                digest = hashlib.sha256(f'{w},{h}:'.encode() + im.tobytes()).hexdigest()
            record = dict(image_id=image_id, subset=subset, filename=p.name,
                          width=w, height=h, pixel_sha256=digest)
            if subset == 'train':
                if image_id not in lookup.index:
                    raise ValueError(f'Missing label: {p.name}')
                x, y = lookup.loc[image_id, ['Fovea_X', 'Fovea_Y']].astype(float)
                if not np.isfinite([x, y]).all() or not (0 <= x < w and 0 <= y < h):
                    raise ValueError(f'Coordinate outside image: {p.name}')
                record.update(x=float(x), y=float(y), nx=float(x / w), ny=float(y / h))
                xml_path = data / 'train_location' / f'{image_id:04}.xml'
                if not xml_path.is_file():
                    raise ValueError(f'Missing XML: {xml_path}')
                xml = ET.parse(xml_path).getroot()
                if (int(xml.findtext('size/width')), int(xml.findtext('size/height'))) != (w, h):
                    raise ValueError(f'XML size mismatch: {p.name}')
                objects = [v for v in xml.findall('object') if v.findtext('name').lower() == 'fovea']
                if len(objects) != 1:
                    raise ValueError(f'Expected one Fovea box: {xml_path}')
                box = objects[0].find('bndbox')
                xmin, ymin, xmax, ymax = [float(box.findtext(k)) for k in ['xmin', 'ymin', 'xmax', 'ymax']]
                if not (xmin <= x <= xmax and ymin <= y <= ymax):
                    raise ValueError(f'CSV coordinate outside XML box: {p.name}')
                record.update(xml_center_dx=float(x - (xmin + xmax) / 2),
                              xml_center_dy=float(y - (ymin + ymax) / 2))
            records.append(record)
    manifest = pd.DataFrame(records)
    if set(manifest.loc[manifest.subset == 'train', 'image_id']) != set(labels['data']):
        raise ValueError('Training images and CSV IDs differ.')
    if manifest.image_id.duplicated().any():
        raise ValueError('Image IDs overlap between train/test.')
    cross = manifest.groupby('pixel_sha256').subset.nunique()
    if (cross > 1).any():
        raise ValueError('Identical pixels occur in both train and test; inspect before continuing.')
    return manifest


def assign_folds(train, seed=42, folds=5):
    """Balance exact-duplicate groups across image resolutions; no test labels used."""
    if folds < 2:
        raise ValueError('Need at least two folds.')
    train = train.copy()
    groups = list(train.groupby('pixel_sha256', sort=True))
    rng = np.random.default_rng(seed)
    buckets = defaultdict(list)
    for digest, group in groups:
        row = group.iloc[0]
        buckets[(int(row.width), int(row.height))].append((digest, len(group)))
    total = np.zeros(folds, dtype=int)
    mapping = {}
    for key in sorted(buckets):
        entries = buckets[key]
        rng.shuffle(entries)
        entries.sort(key=lambda pair: -pair[1])
        sizes = np.zeros(folds, dtype=int)
        for digest, count in entries:
            tie = rng.random(folds)
            fold = min(range(folds), key=lambda f: (sizes[f], total[f], tie[f]))
            mapping[digest] = fold
            sizes[fold] += count
            total[fold] += count
    train['fold'] = train.pixel_sha256.map(mapping)
    if train.groupby('pixel_sha256').fold.nunique().max() != 1:
        raise AssertionError('Duplicate group split across folds.')
    return train


def error_metrics(predicted, truth, sizes):
    delta = (np.asarray(predicted, dtype=np.float64) - np.asarray(truth, dtype=np.float64)) * sizes
    distance = np.linalg.norm(delta, axis=1)
    return dict(n=len(distance), mean_distance_px=float(distance.mean()),
                median_distance_px=float(np.median(distance)),
                p90_distance_px=float(np.quantile(distance, .9)),
                mae_x_px=float(np.abs(delta[:, 0]).mean()),
                mae_y_px=float(np.abs(delta[:, 1]).mean()),
                rmse_coordinate_px=float(np.sqrt(np.mean(delta ** 2))))


class FoveaDataset(Dataset):
    def __init__(self, data, records, image_size, augment=False):
        self.records = records.reset_index(drop=True)
        self.image_size = image_size
        self.augment = augment
        self.images = []
        self.jitter = transforms.ColorJitter(brightness=.2, contrast=.2, saturation=.12, hue=.02)
        self.tensor = transforms.Compose([transforms.ToTensor(),
                                          transforms.Normalize([.485, .456, .406], [.229, .224, .225])])
        for row in self.records.itertuples():
            with Image.open(data / row.subset / row.filename) as im:
                self.images.append(im.convert('RGB').resize((image_size, image_size), Image.Resampling.BILINEAR))

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records.iloc[index]
        im = self.images[index].copy()
        point = (np.array([row.nx, row.ny], dtype=np.float32) if row['subset'] == 'train'
                 else np.array([.5, .5], dtype=np.float32))
        if self.augment:
            if random.random() < .5:
                im = im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
                point[0] = 1 - point[0]
            angle, scale = math.radians(random.uniform(-10, 10)), random.uniform(.92, 1.08)
            a, b = scale * math.cos(angle), -scale * math.sin(angle)
            size, center = self.image_size, self.image_size / 2
            tx, ty = random.uniform(-.035, .035) * size, random.uniform(-.035, .035) * size
            matrix = np.array([[a, b, center + tx - a * center - b * center],
                               [-b, a, center + ty + b * center - a * center], [0, 0, 1]])
            point = ((matrix @ np.array([point[0] * size, point[1] * size, 1]))[:2] / size).astype(np.float32)
            if not (point >= 0).all() or not (point <= 1).all():
                raise ValueError('Augmentation moved label outside image.')
            inverse = np.linalg.inv(matrix)[:2].reshape(-1)
            im = im.transform((size, size), Image.Transform.AFFINE, inverse,
                              resample=Image.Resampling.BILINEAR, fillcolor=(0, 0, 0))
            im = self.jitter(im)
        return self.tensor(im), torch.from_numpy(point), torch.tensor([row.width, row.height], dtype=torch.float32)


def pretrained_backbone(pretrained=True, explicit=None):
    model = models.resnet18(weights=None)
    if not pretrained:
        return model
    base = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
    candidates = [Path(explicit)] if explicit else []
    candidates += [base.parent / 'task2/.torch/hub/checkpoints/resnet18-f37072fd.pth',
                   Path.cwd().parent / 'task2/.torch/hub/checkpoints/resnet18-f37072fd.pth',
                   Path.cwd() / 'task2/.torch/hub/checkpoints/resnet18-f37072fd.pth',
                   Path(torch.hub.get_dir()) / 'checkpoints/resnet18-f37072fd.pth']
    for path in candidates:
        if path.is_file():
            if not hashlib.sha256(path.read_bytes()).hexdigest().startswith('f37072fd'):
                raise ValueError(f'ImageNet checkpoint hash mismatch: {path}')
            model.load_state_dict(torch.load(path, map_location='cpu', weights_only=True))
            print(f'ImageNet weights: {path}', flush=True)
            return model
    # Official torchvision ImageNet weights only. No eye-dataset or Task2 fine-tuned weights.
    state = models.ResNet18_Weights.IMAGENET1K_V1.get_state_dict(progress=True, check_hash=True)
    model.load_state_dict(state)
    return model


class DecoderBlock(nn.Module):
    def __init__(self, incoming, skip, outgoing):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(incoming + skip, outgoing, 3, padding=1, bias=False),
                                    nn.GroupNorm(8, outgoing), nn.ReLU(inplace=True),
                                    nn.Conv2d(outgoing, outgoing, 3, padding=1, bias=False),
                                    nn.GroupNorm(8, outgoing), nn.ReLU(inplace=True))

    def forward(self, value, skip):
        value = F.interpolate(value, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.layers(torch.cat([value, skip], dim=1))


class FoveaHeatmap(nn.Module):
    def __init__(self, pretrained=True, weights=None):
        super().__init__()
        self.backbone = pretrained_backbone(pretrained, weights)
        self.backbone.fc = nn.Identity()
        self.d3 = DecoderBlock(512, 256, 128)
        self.d2 = DecoderBlock(128, 128, 64)
        self.d1 = DecoderBlock(64, 64, 32)
        self.head = nn.Conv2d(32, 1, 1)
        self.freeze_bn = pretrained

    def train(self, mode=True):
        super().train(mode)
        if mode and self.freeze_bn:
            for module in self.backbone.modules():
                if isinstance(module, nn.BatchNorm2d):
                    module.eval()
        return self

    def forward(self, image):
        b = self.backbone
        value = b.maxpool(b.relu(b.bn1(b.conv1(image))))
        c1 = b.layer1(value)
        c2 = b.layer2(c1)
        c3 = b.layer3(c2)
        c4 = b.layer4(c3)
        return self.head(self.d1(self.d2(self.d3(c4, c3), c2), c1))[:, 0]


def heatmap_loss(logits, points, sigma=1.5):
    height, width = logits.shape[-2:]
    yy = torch.arange(height, device=logits.device, dtype=torch.float32) + .5
    xx = torch.arange(width, device=logits.device, dtype=torch.float32) + .5
    distance = (xx[None, None, :] - points[:, 0, None, None] * width) ** 2
    distance = distance + (yy[None, :, None] - points[:, 1, None, None] * height) ** 2
    target = torch.exp(-distance / (2 * sigma ** 2))
    target = target / target.sum(dim=(-2, -1), keepdim=True).clamp_min(1e-12)
    return -(target.flatten(1) * F.log_softmax(logits.float().flatten(1), dim=1)).sum(1).mean()


def decode_heatmap(logits, radius=3):
    """Local expectation around the peak; cell centers match resize align_corners=False."""
    batch, height, width = logits.shape
    peak = logits.flatten(1).argmax(1)
    py, px = peak // width, peak % width
    yy = torch.arange(height, device=logits.device)[None, :, None]
    xx = torch.arange(width, device=logits.device)[None, None, :]
    mask = (yy - py[:, None, None]).abs() <= radius
    mask = mask & ((xx - px[:, None, None]).abs() <= radius)
    weights = logits.float().masked_fill(~mask, -torch.inf).flatten(1).softmax(1).view(batch, height, width)
    x = (weights * ((xx + .5) / width)).sum((-2, -1))
    y = (weights * ((yy + .5) / height)).sum((-2, -1))
    return torch.stack([x, y], dim=1)


@torch.inference_mode()
def predict(model, loader, device, flip_tta=False):
    model.eval()
    predictions = []
    for images, _, _ in loader:
        images = images.to(device)
        with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
            points = decode_heatmap(model(images))
            if flip_tta:
                other = decode_heatmap(model(images.flip(-1)))
                other[:, 0] = 1 - other[:, 0]
                points = (points + other) / 2
        predictions.append(points.cpu().numpy())
    return np.concatenate(predictions)


@contextmanager
def training_lock(out):
    handle = (out / 'training.lock').open('a+b')
    if handle.tell() == 0:
        handle.write(b'0')
        handle.flush()
    handle.seek(0)
    try:
        if os.name == 'nt':
            import msvcrt
            msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
        else:
            import fcntl
            fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        yield
    finally:
        handle.close()


def train_model(data, training, validation, out, config, device, model_builder=None, dataset_builder=None):
    out.mkdir(parents=True, exist_ok=True)
    with training_lock(out):
        # Atomic checkpoint replacement temporarily keeps both the old and new files.
        if shutil.disk_usage(out).free < 600 * 1024 ** 2:
            raise RuntimeError(f'Need at least 600 MiB free for atomic checkpoints; choose another output directory: {out}')
        config_path = out / 'config.json'
        if config_path.exists() and json.loads(config_path.read_text(encoding='utf-8')) != config:
            raise ValueError(f'Configuration changed. Use a new output directory: {out}')
        save_json(config_path, config)
        seed_everything(config['seed'])
        dataset_builder = dataset_builder or FoveaDataset
        train_set = dataset_builder(data, training, config['image_size'], augment=True)
        loader = DataLoader(train_set, batch_size=config['batch_size'], shuffle=True, num_workers=0)
        valid_loader = (DataLoader(dataset_builder(data, validation, config['image_size']),
                                  batch_size=config['batch_size'], shuffle=False, num_workers=0)
                        if validation is not None else None)
        model = (model_builder(config) if model_builder else FoveaHeatmap(config['pretrained'], config['weights'])).to(device)
        backbone_ids = {id(v) for v in model.backbone.parameters()}
        decoder_params = [v for v in model.parameters() if id(v) not in backbone_ids]
        optimizer = torch.optim.AdamW([{'params': model.backbone.parameters(), 'lr': config['lr_backbone']},
                                       {'params': decoder_params, 'lr': config['lr_decoder']}],
                                      weight_decay=config['weight_decay'])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.get('schedule_epochs', config['epochs']),
                                                               eta_min=config['lr_backbone'] * .05)
        scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
        start, best, best_epoch, history = 0, float('inf'), 0, []
        last_path, best_path = out / 'last.pt', out / 'best.pt'
        if last_path.exists():
            checkpoint = torch.load(last_path, map_location='cpu', weights_only=False)
            model.load_state_dict(checkpoint['model'])
            optimizer.load_state_dict(checkpoint['optimizer'])
            scheduler.load_state_dict(checkpoint['scheduler'])
            scaler.load_state_dict(checkpoint['scaler'])
            start, best, best_epoch, history = (checkpoint[k] for k in ['epoch', 'best', 'best_epoch', 'history'])
            random.setstate(checkpoint['python_rng'])
            np.random.set_state(checkpoint['numpy_rng'])
            torch.set_rng_state(checkpoint['torch_rng'])
            if device.type == 'cuda':
                torch.cuda.set_rng_state_all(checkpoint['cuda_rng'])
            print(f'Resume {out.name}: epoch {start}', flush=True)
        for epoch in range(start, config['epochs']):
            begin = time.perf_counter()
            model.train()
            total, seen = 0., 0
            for images, points, _ in loader:
                images, points = images.to(device), points.to(device)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
                    loss = heatmap_loss(model(images), points, config['sigma'])
                if not torch.isfinite(loss):
                    raise RuntimeError('Non-finite training loss.')
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.)
                scaler.step(optimizer)
                scaler.update()
                total += float(loss.detach()) * len(images)
                seen += len(images)
            row = dict(epoch=epoch + 1, train_heatmap_ce=total / seen, lr_backbone=optimizer.param_groups[0]['lr'])
            if valid_loader is not None:
                prediction = predict(model, valid_loader, device)
                row.update(error_metrics(prediction, validation[['nx', 'ny']].to_numpy(),
                                         validation[['width', 'height']].to_numpy()))
                score = row['mean_distance_px']
            else:
                score = 0.
            if score < best or valid_loader is None:
                best, best_epoch = score, epoch + 1
                torch.save(dict(model=model.state_dict(), epoch=best_epoch, config=config), out / 'best.tmp')
                (out / 'best.tmp').replace(best_path)
            scheduler.step()
            row['seconds'] = time.perf_counter() - begin
            history.append(row)
            checkpoint = dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
                              scheduler=scheduler.state_dict(), scaler=scaler.state_dict(), epoch=epoch + 1,
                              best=best, best_epoch=best_epoch, history=history,
                              python_rng=random.getstate(), numpy_rng=np.random.get_state(),
                              torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all())
            torch.save(checkpoint, out / 'last.tmp')
            (out / 'last.tmp').replace(last_path)
            pd.DataFrame(history).to_csv(out / 'history.csv', index=False)
            save_json(out / 'status.json', dict(state='running', epoch=epoch + 1, epochs=config['epochs'],
                                               best_epoch=best_epoch, best_mean_distance_px=best,
                                               pid=os.getpid(), updated_unix=time.time()))
            print(f'{out.name} {epoch+1:03}/{config["epochs"]} ce={row["train_heatmap_ce"]:.4f}'
                  + (f' val={score:.2f}px best={best:.2f}px@{best_epoch}' if valid_loader else '')
                  + f' {row["seconds"]:.1f}s', flush=True)
        checkpoint = torch.load(best_path, map_location='cpu', weights_only=False)
        model.load_state_dict(checkpoint['model'])
        save_json(out / 'status.json', dict(state='completed', best_epoch=best_epoch,
                                           best_mean_distance_px=best if validation is not None else None,
                                           updated_unix=time.time()))
        return model, best_epoch

def stable_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     separators=(',', ':'), allow_nan=False).encode('utf-8')).hexdigest()


def atomic_csv(path, rows, fields=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = list(rows)
    fields = list(fields or (list(rows[0]) if rows else []))
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def read_rows(path):
    with Path(path).open(encoding='utf-8', newline='') as file:
        return list(csv.DictReader(file))


def prediction_rows(records, points, fold, labeled):
    if len(records) != len(points):
        raise ValueError('Prediction count differs from image count.')
    rows, seen = [], set()
    for item, point in zip(records, points):
        key, width, height = int(item['image_id']), int(item['width']), int(item['height'])
        if key in seen:
            raise ValueError('Duplicate image ID.')
        seen.add(key)
        nx, ny = float(point[0]), float(point[1])
        if not all(math.isfinite(v) for v in [nx, ny]) or not (0 <= nx < 1 and 0 <= ny < 1):
            raise ValueError(f'Invalid normalized prediction for image {key}.')
        row = dict(image_id=key, fold=int(fold), width=width, height=height,
                   pixel_sha256=str(item['pixel_sha256']), nx=nx, ny=ny,
                   Fovea_X=nx * width, Fovea_Y=ny * height)
        if labeled:
            x, y = float(item['x']), float(item['y'])
            dx, dy = row['Fovea_X'] - x, row['Fovea_Y'] - y
            row.update(gt_x=x, gt_y=y, dx=dx, dy=dy,
                       distance_px=math.hypot(dx, dy), squared_error=dx * dx + dy * dy)
        rows.append(row)
    return rows


def quantile_linear(values, probability):
    values = sorted(float(v) for v in values)
    if not values:
        raise ValueError('No values.')
    position = (len(values) - 1) * probability
    low, high = math.floor(position), math.ceil(position)
    return values[low] + (values[high] - values[low]) * (position - low)


def metrics_from_rows(rows):
    rows = list(rows)
    if not rows:
        raise ValueError('No labeled predictions.')
    distance, dx, dy = [], [], []
    for row in rows:
        ex = float(row['Fovea_X']) - float(row['gt_x'])
        ey = float(row['Fovea_Y']) - float(row['gt_y'])
        if not math.isfinite(ex) or not math.isfinite(ey):
            raise ValueError('Non-finite error.')
        dx.append(ex)
        dy.append(ey)
        distance.append(math.hypot(ex, ey))
    return dict(n=len(rows), mean_distance_px=statistics.mean(distance),
                median_distance_px=statistics.median(distance),
                p90_distance_px=quantile_linear(distance, .9), max_distance_px=max(distance),
                mae_x_px=statistics.mean(abs(v) for v in dx), mae_y_px=statistics.mean(abs(v) for v in dy),
                rmse_coordinate_px=math.sqrt(sum(x*x+y*y for x, y in zip(dx, dy)) / (2 * len(rows))))


def validate_oof(rows, records):
    expected = {int(r['image_id']): r for r in records}
    if len(expected) != len(records) or len(rows) != len(records):
        raise ValueError('OOF must contain exactly one row per training image.')
    seen, groups = set(), {}
    for row in rows:
        key = int(row['image_id'])
        if key not in expected or key in seen:
            raise ValueError('Unknown or repeated OOF image ID.')
        seen.add(key)
        source = expected[key]
        if int(row['fold']) != int(source['fold']):
            raise ValueError('OOF fold does not match the fixed split.')
        if (int(row['width']), int(row['height'])) != (int(source['width']), int(source['height'])):
            raise ValueError('OOF dimensions differ.')
        if str(row['pixel_sha256']) != str(source['pixel_sha256']):
            raise ValueError('OOF image hash differs.')
        if abs(float(row['gt_x']) - float(source['x'])) > 1e-6 or abs(float(row['gt_y']) - float(source['y'])) > 1e-6:
            raise ValueError('OOF ground truth differs.')
        groups.setdefault(str(source['pixel_sha256']), set()).add(int(row['fold']))
    if seen != set(expected) or any(len(value) != 1 for value in groups.values()):
        raise ValueError('Incomplete OOF coverage or duplicate group leakage.')
    return sorted(rows, key=lambda r: int(r['image_id']))


def validate_test_rows(rows, records, fold):
    expected = {int(r['image_id']): r for r in records}
    if len(rows) != len(expected):
        raise ValueError('Incomplete test prediction coverage.')
    seen = set()
    for row in rows:
        key = int(row['image_id'])
        if key not in expected or key in seen or int(row['fold']) != fold:
            raise ValueError('Invalid test ID/fold.')
        seen.add(key)
        source = expected[key]
        if (int(row['width']), int(row['height'])) != (int(source['width']), int(source['height'])):
            raise ValueError('Test image dimensions differ.')
        if str(row['pixel_sha256']) != str(source['pixel_sha256']):
            raise ValueError('Test image hash differs.')
        x, y = float(row['Fovea_X']), float(row['Fovea_Y'])
        if not all(math.isfinite(v) for v in [x, y]) or not (0 <= x < int(source['width']) and 0 <= y < int(source['height'])):
            raise ValueError('Test prediction outside image.')
    return {int(r['image_id']): r for r in rows}


def average_test_folds(folds, records, fold_count=5):
    if set(folds) != set(range(fold_count)):
        raise ValueError('All five folds are required before exporting a five-fold submission.')
    mappings = [validate_test_rows(folds[f], records, f) for f in range(fold_count)]
    averaged = []
    for source in records:
        key = int(source['image_id'])
        averaged.append(dict(image_id=key, width=int(source['width']), height=int(source['height']),
                             Fovea_X=statistics.mean(float(m[key]['Fovea_X']) for m in mappings),
                             Fovea_Y=statistics.mean(float(m[key]['Fovea_Y']) for m in mappings)))
    return averaged


def write_submission(template_path, predictions, target):
    lookup = {}
    for row in predictions:
        key = int(row['image_id'])
        for axis, dimension in [('X', 'width'), ('Y', 'height')]:
            value = float(row[f'Fovea_{axis}'])
            if not math.isfinite(value) or not 0 <= value < int(row[dimension]):
                raise ValueError('Invalid original-pixel coordinate.')
            image_id = f'{key}_Fovea_{axis}'
            if image_id in lookup:
                raise ValueError('Repeated submission ID.')
            lookup[image_id] = value
    template = read_rows(template_path)
    ids = [row['ImageID'] for row in template]
    if len(ids) != len(set(ids)) or set(ids) != set(lookup):
        raise ValueError('Submission IDs differ from template.')
    rows = [dict(ImageID=key, value=f'{lookup[key]:.6f}') for key in ids]
    atomic_csv(target, rows, ['ImageID', 'value'])
    check = read_rows(target)
    if [r['ImageID'] for r in check] != ids or any(not math.isfinite(float(r['value'])) for r in check):
        raise AssertionError('Submission round trip failed.')


def resolve_weights(output, explicit=None):
    candidates = [Path(explicit)] if explicit else []
    candidates += [Path(torch.hub.get_dir()) / 'checkpoints' / WEIGHT_NAME]
    if '__file__' in globals():
        candidates.append(Path(__file__).resolve().parent.parent / 'task2/.torch/hub/checkpoints' / WEIGHT_NAME)
    if Path('/kaggle/input').exists():
        candidates += sorted(Path('/kaggle/input').rglob(WEIGHT_NAME))
    path = next((p for p in candidates if p.is_file()), None)
    if path is None:
        path = output / '_pretrained' / WEIGHT_NAME
        path.parent.mkdir(parents=True, exist_ok=True)
        if not path.is_file():
            print('Downloading official ResNet34 ImageNet weights (about 87 MB).', flush=True)
            torch.hub.download_url_to_file(WEIGHT_URL, str(path), hash_prefix='b627a593', progress=True)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if not digest.startswith('b627a593'):
        raise ValueError(f'ImageNet weight hash mismatch: {path}')
    save_json(output / 'pretrained_source.json', dict(path=str(path), url=WEIGHT_URL, sha256=digest))
    return path


class ResNet34CV(nn.Module):
    def __init__(self, weights=None):
        super().__init__()
        self.backbone = models.resnet34(weights=None)
        if weights is not None:
            self.backbone.load_state_dict(torch.load(weights, map_location='cpu', weights_only=True), strict=True)
        self.backbone.fc = nn.Identity()
        self.d3 = DecoderBlock(512, 256, 128)
        self.d2 = DecoderBlock(128, 128, 64)
        self.d1 = DecoderBlock(64, 64, 32)
        self.head = nn.Conv2d(32, 1, 1)

    def train(self, mode=True):
        super().train(mode)
        if mode:
            for module in self.backbone.modules():
                if isinstance(module, nn.BatchNorm2d):
                    module.eval()
        return self

    def forward(self, images):
        b = self.backbone
        x = b.maxpool(b.relu(b.bn1(b.conv1(images))))
        c1 = b.layer1(x)
        c2 = b.layer2(c1)
        c3 = b.layer3(c2)
        c4 = b.layer4(c3)
        return self.head(self.d1(self.d2(self.d3(c4, c3), c2), c1))[:, 0]


@torch.inference_mode()
def predict_cv(model, loader, device, radius, flip=False):
    model.eval()
    result = []
    for images, _, _ in loader:
        images = images.to(device)
        with torch.autocast(device_type='cuda', enabled=True):
            coordinates = decode_heatmap(model(images), radius=radius)
            if flip:
                mirrored = decode_heatmap(model(images.flip(-1)), radius=radius)
                mirrored[:, 0] = 1 - mirrored[:, 0]
                coordinates = (coordinates + mirrored) / 2
        result.extend(coordinates.float().cpu().tolist())
    return result


def atomic_torch_save(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(value, temporary)
    temporary.replace(path)


def restore_backup(source, output, protocol):
    if source is None:
        return
    source = Path(source).resolve()
    if source == output.resolve():
        return
    candidate = source / 'protocol.json'
    if not candidate.is_file() or json.loads(candidate.read_text()) != protocol:
        raise ValueError('Backup protocol/data/code differs. Choose the correct prior output folder.')
    if (output / 'protocol.json').exists():
        print('Current output already has progress; resuming it without merging backup files.', flush=True)
        return
    for path in source.rglob('*'):
        if not path.is_file() or path.is_symlink() or path.suffix in {'.tmp', '.zip'}:
            continue
        if not path.resolve().is_relative_to(source):
            raise ValueError('Backup path escapes its source directory.')
        destination = output / path.relative_to(source)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, destination)

class CoarseBundle:
    """Read only metadata/predictions from a directory or ZIP; never extract/load weights."""
    def __init__(self, path):
        self.path, self.prefix, self.archive = Path(path).resolve(), '', None
        if self.path.is_dir():
            return
        if not self.path.is_file() or self.path.suffix.lower() != '.zip':
            raise ValueError(f'Coarse source is not a directory or ZIP: {path}')
        self.archive = zipfile.ZipFile(self.path)
        try:
            candidates = []
            for entry in self.archive.infolist():
                pure = PurePosixPath(entry.filename)
                if pure.is_absolute() or '..' in pure.parts or '\\' in entry.filename:
                    raise ValueError('Unsafe archive member path.')
                if pure.name == 'protocol.json' and entry.file_size < 1000000:
                    value = json.loads(self.archive.read(entry).decode('utf-8-sig'))
                    if value.get('version') == 'resnet34-cv-v1':
                        candidates.append(entry.filename[:-len('protocol.json')])
            if len(candidates) != 1:
                raise ValueError('ZIP must contain exactly one original CV protocol.')
            self.prefix = candidates[0]
        except BaseException:
            self.close()
            raise

    def text(self, relative):
        if self.archive is None:
            path = (self.path / relative).resolve()
            if not path.is_relative_to(self.path):
                raise ValueError('Coarse file escapes its source folder.')
            if path.stat().st_size > 5000000:
                raise ValueError('Unexpectedly large metadata file.')
            return path.read_text(encoding='utf-8-sig')
        name = self.prefix + relative
        if self.archive.getinfo(name).file_size > 5000000:
            raise ValueError('Unexpectedly large metadata member.')
        return self.archive.read(name).decode('utf-8-sig')

    def rows(self, name):
        return list(csv.DictReader(io.StringIO(self.text(name), newline='')))

    def json(self, name):
        return json.loads(self.text(name))

    def close(self):
        if self.archive is not None:
            self.archive.close()
            self.archive = None


def validate_coarse_bundle(source, train_records, test_records):
    bundle = CoarseBundle(source)
    try:
        protocol = bundle.json('protocol.json')
        if protocol.get('version') != 'resnet34-cv-v1' or protocol.get('folds') != 5:
            raise ValueError('Expected the previous five-fold CV experiment.')
        if protocol.get('selection_metric') != 'rmse_coordinate_px':
            raise ValueError('This predeclared experiment reuses the RMSE-selected 512 coarse checkpoints.')
        old_split = bundle.rows('split.csv')
        if len(old_split) != len(train_records):
            raise ValueError('Previous split is incomplete.')
        prior = {int(r['image_id']): r for r in old_split}
        if len(prior) != len(train_records):
            raise ValueError('Duplicate IDs in previous split.')
        for record in train_records:
            old = prior[int(record['image_id'])]
            for field in ('fold', 'width', 'height'):
                if int(old[field]) != int(record[field]):
                    raise ValueError('Coarse split or dimensions differ.')
            if old['pixel_sha256'] != record['pixel_sha256']:
                raise ValueError('Coarse image hash differs.')
            if any(abs(float(old[k])-float(record[k])) > 1e-6 for k in ('x', 'y')):
                raise ValueError('Coarse labels differ.')
        validation, testing, metadata, coverage = {}, {}, [], []
        for fold in range(5):
            prefix = f'resnet34_512/fold_{fold}/'
            config = bundle.json(prefix+'train_config.json')
            done = bundle.json(prefix+'fold_complete.json')
            signature = stable_hash(config)
            if (int(config['fold']) != fold or int(config['image_size']) != 512 or
                    config['data_fingerprint'] != protocol['data_fingerprint'] or
                    config['source_sha256'] != protocol['source_sha256'] or
                    done.get('signature') != signature or not done.get('predictions_complete') or
                    done['training']['signature'] != signature):
                raise ValueError('Coarse fold metadata is inconsistent.')
            held = [r for r in train_records if int(r['fold']) == fold]
            rows = validate_oof(bundle.rows(prefix+'validation_raw.csv'), held)
            # Verify finite in-bounds coordinates as well as OOF membership/labels.
            validate_test_rows(rows, held, fold)
            best = done['training']['best']['rmse_coordinate_px']
            if not math.isclose(metrics_from_rows(rows)['rmse_coordinate_px'], float(best['score']), abs_tol=1e-6):
                raise ValueError('Coarse predictions do not match the selected checkpoint metrics.')
            validation[fold] = {int(r['image_id']): r for r in rows}
            testing[fold] = validate_test_rows(bundle.rows(prefix+'test_raw.csv'), test_records, fold)
            metadata.append(dict(fold=fold, config=config, selected=best))
            coverage.extend(rows)
        fingerprint = stable_hash(dict(protocol=protocol, split=old_split, folds=metadata,
                                        validation=validation, test=testing))
        return dict(validation=validation, test=testing, fingerprint=fingerprint,
                    protocol=protocol, metadata=metadata,
                    baseline_metrics=metrics_from_rows(coverage))
    finally:
        bundle.close()


def crop_geometry(center, side):
    if side < 1 or int(side) != side or not all(math.isfinite(float(v)) for v in center):
        raise ValueError('Invalid crop geometry.')
    left, top = [math.floor(float(v)-side/2) for v in center]
    return (left, top, left+int(side), top+int(side))


def point_to_patch(point, box):
    return ((float(point[0])-box[0])/(box[2]-box[0]),
            (float(point[1])-box[1])/(box[3]-box[1]))


def patch_to_original(point, box):
    return (box[0]+float(point[0])*(box[2]-box[0]),
            box[1]+float(point[1])*(box[3]-box[1]))


def make_refinement_row(record, point, coarse, box, fold, max_shift):
    """Map patch coordinates back before scoring; a fixed fallback needs no labels."""
    width, height = int(record['width']), int(record['height'])
    cx, cy = float(coarse['Fovea_X']), float(coarse['Fovea_Y'])
    if not (0 <= cx < width and 0 <= cy < height):
        raise ValueError('Coarse coordinate outside original image.')
    raw_x, raw_y = patch_to_original(point, box)
    shift = math.hypot(raw_x-cx, raw_y-cy)
    reason = ''
    if not all(math.isfinite(v) for v in (raw_x, raw_y)):
        raise ValueError('Non-finite refinement coordinate.')
    if not (0 <= raw_x < width and 0 <= raw_y < height):
        reason = 'outside_image'
    elif shift > max_shift:
        reason = 'shift_gt_fixed_limit'
    x, y = (cx, cy) if reason else (raw_x, raw_y)
    row = dict(image_id=int(record['image_id']), fold=int(fold), width=width, height=height,
               pixel_sha256=str(record['pixel_sha256']), nx=x/width, ny=y/height,
               Fovea_X=x, Fovea_Y=y, coarse_x=cx, coarse_y=cy,
               raw_refined_x=raw_x, raw_refined_y=raw_y, shift_px=shift,
               fallback=int(bool(reason)), fallback_reason=reason,
               crop_left=box[0], crop_top=box[1], crop_side=box[2]-box[0])
    if record['subset'] == 'train':
        dx, dy = x-float(record['x']), y-float(record['y'])
        row.update(gt_x=float(record['x']), gt_y=float(record['y']), dx=dx, dy=dy,
                   distance_px=math.hypot(dx, dy), squared_error=dx*dx+dy*dy)
    return row


def blend_prediction_rows(coarse_rows, refine_rows, labeled):
    coarse = {int(r['image_id']): r for r in coarse_rows}
    refined = {int(r['image_id']): r for r in refine_rows}
    if len(coarse) != len(coarse_rows) or len(refined) != len(refine_rows) or set(coarse) != set(refined):
        raise ValueError('Blend IDs differ or repeat.')
    result = []
    for key in sorted(coarse):
        a, b = coarse[key], refined[key]
        for field in ('fold', 'width', 'height'):
            if int(a[field]) != int(b[field]):
                raise ValueError('Blend fold or dimensions differ.')
        if a['pixel_sha256'] != b['pixel_sha256']:
            raise ValueError('Blend pixels differ.')
        row = {name:a[name] for name in ('image_id', 'fold', 'width', 'height', 'pixel_sha256')}
        x, y = [(float(a[f'Fovea_{axis}'])+float(b[f'Fovea_{axis}']))/2 for axis in ('X','Y')]
        row.update(Fovea_X=x, Fovea_Y=y, nx=x/int(a['width']), ny=y/int(a['height']))
        if labeled:
            if any(abs(float(a[k])-float(b[k])) > 1e-6 for k in ('gt_x','gt_y')):
                raise ValueError('Blend ground truth differs.')
            dx, dy = x-float(a['gt_x']), y-float(a['gt_y'])
            row.update(gt_x=float(a['gt_x']), gt_y=float(a['gt_y']), dx=dx, dy=dy,
                       distance_px=math.hypot(dx,dy), squared_error=dx*dx+dy*dy)
        result.append(row)
    return result


class OriginalImageCache:
    def __init__(self, data):
        self.data, self.images = Path(data), {}

    def get(self, row):
        key = int(row['image_id'])
        if key not in self.images:
            with Image.open(self.data/row['subset']/row['filename']) as image:
                self.images[key] = image.convert('RGB')
        image = self.images[key]
        if image.size != (int(row['width']), int(row['height'])):
            raise ValueError('Original image dimensions changed.')
        return image


# Fixed coarse predictions from the previous run; no coarse weights or new labels embedded.
COARSE_CACHE_B64='...'
COARSE_CACHE_SHA256='0fc2ee733b1ec10d91d2fe1800c0224c2b098f408eaaf4b543a16722135b4fd6'
if COARSE_SOURCE is None:
    _data=base64.b64decode(COARSE_CACHE_B64,validate=True)
    if hashlib.sha256(_data).hexdigest()!=COARSE_CACHE_SHA256:raise RuntimeError('内置粗预测校验失败。')
    _path=Path('/kaggle/working/task3_coarse512_v3_cache.zip')
    if not _path.is_file() or hashlib.sha256(_path.read_bytes()).hexdigest()!=COARSE_CACHE_SHA256:_path.write_bytes(_data)
    COARSE_SOURCE=_path
    del _data
    print('已载入固定的512粗预测（公开对照14.16538）。')


In [ ]:
from __future__ import annotations
DIAGNOSTIC_SOURCE_SHA256={'task3_pipeline.py': '5a27db4b7901d185eec831080a16d783ad001adc22b639474d345fe7b315ec3f', 'task3_kaggle_cv.py': '2227703a35b3955b8ce02801930b0bf496703decece4ad5234a1d1f70a0b12e6', 'task3_kaggle_refine.py': 'feed93ec81a810165a968715b55155f381f987f13920777819a688e55e8822cb', 'task3_refine_diagnostic_v3.py': '72387a1d8c0636767aa3881f636c4241136ce52367d49ebd8301fabeb9b0cf26'}
"""Conditional refinement ablations: inner-only stopping and fixed coarse outputs."""
import csv
import gc
import hashlib
import json
import math
from pathlib import Path
import random
import shutil
import statistics
import sys
import time
import zipfile


import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

DIAGNOSTIC_VERSION='refine-diagnostic-v3'
VARIANTS=('wide','small','small_prior')


def diagnostic_sources():
    if 'DIAGNOSTIC_SOURCE_SHA256' in globals():return dict(DIAGNOSTIC_SOURCE_SHA256)
    base=Path(__file__).resolve().parent
    return {name:hashlib.sha256((base/name).read_bytes()).hexdigest() for name in
        ('task3_pipeline.py','task3_kaggle_cv.py','task3_kaggle_refine.py','task3_refine_diagnostic_v3.py')}


def inner_partition(train,outer_fold):
    development=train[train.fold!=outer_fold].copy()
    outer=train[train.fold==outer_fold].copy()
    assigned=assign_folds(development,7000+outer_fold,5)
    validation_ids=set(int(x) for x in assigned.loc[assigned.fold==0,'image_id'])
    fit=development[~development.image_id.isin(validation_ids)].copy()
    validation=development[development.image_id.isin(validation_ids)].copy()
    sets=[set(frame.pixel_sha256) for frame in (fit,validation,outer)]
    if not len(fit) or not len(validation) or any(sets[i]&sets[j] for i,j in ((0,1),(0,2),(1,2))):
        raise ValueError('Inner/outer image groups overlap or are empty.')
    if set(fit.image_id)|set(validation.image_id)!=set(development.image_id):
        raise ValueError('Inner split does not cover the outer training images.')
    return development,fit,validation,outer


def center_offset(arm,uniforms):
    u,v,tail=[float(x) for x in uniforms]
    if not all(0<x<1 for x in (u,v,tail)):raise ValueError('Uniforms must be strictly between0 and1.')
    if arm=='wide':return ((2*u-1)*160,(2*v-1)*160)
    if arm!='small':raise ValueError('Unknown training arm.')
    if tail<.1:return ((2*u-1)*64,(2*v-1)*64)
    normal=statistics.NormalDist()
    return (max(-48.,min(48.,normal.inv_cdf(u)*16)),max(-48.,min(48.,normal.inv_cdf(v)*16)))


def selected_inner_epoch(best,baseline,min_gain=.05):
    score=float(best['score']);base=float(baseline)
    if not math.isfinite(score+base) or base<0 or int(best['epoch'])<1:
        raise ValueError('Invalid inner selection metrics.')
    enabled=base>0 and score<=(1-min_gain)*base
    return dict(enabled=enabled,epoch=int(best['epoch']) if enabled else 0,
                inner_best_epoch=int(best['epoch']),inner_score=score,inner_baseline=base,
                min_relative_gain=min_gain,reason='inner_gain_passed' if enabled else 'inner_gain_below_fixed_margin')


def validate_phase_config(config):
    fit=set(config['train_ids']);held=set(config['evaluation_ids']);forbidden=set(config['forbidden_ids'])
    if (len(fit)!=len(config['train_ids']) or len(held)!=len(config['evaluation_ids']) or
            len(forbidden)!=len(config['forbidden_ids']) or not fit or fit&held or fit&forbidden or held&forbidden):
        raise ValueError('Phase IDs overlap or repeat.')
    if config['role'] not in ('inner','outer') or config['training_arm'] not in ('wide','small'):
        raise ValueError('Unknown training role/arm.')
    if config['role']=='inner' and not held:raise ValueError('Inner phase needs independent held-out images.')
    if config['role']=='outer' and held:raise ValueError('Outer refit cannot use outer labels for stopping.')
    if config['epochs']<1 or config['image_size']!=384 or config['crop_side']!=768:
        raise ValueError('Invalid fine phase geometry/epochs.')


def validate_phase_request(config,has_evaluator,snapshots):
    validate_phase_config(config)
    if (config['role']=='inner')!=bool(has_evaluator):
        raise ValueError('Only inner fitting may evaluate a held-out set for epoch selection.')
    if config['role']=='outer' and (not snapshots or max(snapshots)!=config['epochs'] or min(snapshots)<1):
        raise ValueError('Outer refit needs exactly preselected epoch snapshots, including its final epoch.')
    if config['role']=='inner' and snapshots:raise ValueError('Inner checkpoints must be selected by inner metrics.')


class DiagnosticPatchDataset(Dataset):
    def __init__(self,records,cache,config,training,centers=None):
        self.records,self.cache,self.config=list(records),cache,config
        self.training,self.centers=training,centers
        ids={int(r['image_id']) for r in records}
        if len(ids)!=len(records):raise ValueError('Repeated image in patch dataset.')
        if training:
            if ids!=set(config['train_ids']) or ids&set(config['forbidden_ids']):
                raise ValueError('Training dataset differs from allowed phase images.')
        elif centers is None or ids!=set(centers):raise ValueError('Prediction centers/IDs differ.')
        self.tensor=transforms.Compose([transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
        self.jitter=transforms.ColorJitter(brightness=.15,contrast=.15,saturation=.08,hue=.01)

    def __len__(self):return len(self.records)

    def __getitem__(self,index):
        row,cfg=self.records[index],self.config
        if self.training:
            # Both training arms consume the same3 draws before identical augmentations.
            draws=[max(1e-10,min(1-1e-10,random.random())) for _ in range(3)]
            dx,dy=center_offset(cfg['training_arm'],draws)
            center=(float(row['x'])+dx,float(row['y'])+dy)
        else:
            center=(float(self.centers[int(row['image_id'])]['Fovea_X']),float(self.centers[int(row['image_id'])]['Fovea_Y']))
        box=crop_geometry(center,768)
        image=self.cache.get(row).crop(box).resize((384,384),Image.Resampling.BILINEAR)
        point=np.array(point_to_patch((row['x'],row['y']),box) if self.training else (.5,.5),dtype=np.float32)
        if self.training:
            if random.random()<.5:
                image=image.transpose(Image.Transpose.FLIP_LEFT_RIGHT);point[0]=1-point[0]
            angle,scale=math.radians(random.uniform(-8,8)),random.uniform(.95,1.05)
            a,b=scale*math.cos(angle),-scale*math.sin(angle)
            tx,ty=random.uniform(-.02,.02)*384,random.uniform(-.02,.02)*384
            matrix=np.array([[a,b,192+tx-a*192-b*192],[-b,a,192+ty+b*192-a*192],[0,0,1]])
            point=((matrix@[point[0]*384,point[1]*384,1])[:2]/384).astype(np.float32)
            if not np.isfinite(point).all() or not ((point>0).all() and (point<1).all()):
                raise ValueError('Augmentation moved the target out of patch.')
            image=image.transform((384,384),Image.Transform.AFFINE,np.linalg.inv(matrix)[:2].reshape(-1),
                                  resample=Image.Resampling.BILINEAR,fillcolor=(0,0,0))
            image=self.jitter(image)
        return self.tensor(image),torch.from_numpy(point),torch.tensor(box,dtype=torch.int64)


def coarse_log_prior(logits,boxes,coarse_points,sigma_native=48.):
    if sigma_native<=0:raise ValueError('Prior width must be positive.')
    height,width=logits.shape[-2:]
    # Pixel centers use the same convention as decode_heatmap and PIL resize.
    box=boxes.to(logits.device,dtype=torch.float32)
    points=torch.as_tensor(coarse_points,device=logits.device,dtype=torch.float32)
    xs=box[:,0,None,None]+(torch.arange(width,device=logits.device)[None,None,:]+.5)*(768/width)
    ys=box[:,1,None,None]+(torch.arange(height,device=logits.device)[None,:,None]+.5)*(768/height)
    distance=(xs-points[:,0,None,None])**2+(ys-points[:,1,None,None])**2
    return logits.float()-distance/(2*sigma_native**2)


@torch.inference_mode()
def diagnostic_predict(model,loader,config,centers,fold,modes):
    model.eval();outputs={mode:[] for mode in modes};offset=0
    for images,_,boxes in loader:
        with torch.autocast(device_type='cuda',enabled=True):logits=model(images.to('cuda:0'))
        if logits.shape!=(len(images),96,96) or not torch.isfinite(logits).all():
            raise RuntimeError('Invalid diagnostic heatmap.')
        records=loader.dataset.records[offset:offset+len(images)]
        prior_points=[[float(centers[int(r['image_id'])]['Fovea_X']),float(centers[int(r['image_id'])]['Fovea_Y'])] for r in records]
        for mode in modes:
            value=logits if mode=='plain' else coarse_log_prior(logits,boxes,prior_points,config['prior_sigma'])
            points=decode_heatmap(value,radius=3).cpu().tolist()
            for r,point,box in zip(records,points,boxes.tolist()):
                row=make_refinement_row(r,point,centers[int(r['image_id'])],box,fold,256.)
                row.update(decoder=mode,refine_applied=1)
                outputs[mode].append(row)
        offset+=len(images)
    if offset!=len(loader.dataset):raise ValueError('Inference count differs.')
    return outputs


def raw_prediction_rows(rows):
    result=[]
    for source in rows:
        row=dict(source)
        x,y=float(row['raw_refined_x']),float(row['raw_refined_y'])
        row.update(Fovea_X=x,Fovea_Y=y,nx=x/int(row['width']),ny=y/int(row['height']))
        if 'gt_x' in row:
            dx,dy=x-float(row['gt_x']),y-float(row['gt_y'])
            row.update(dx=dx,dy=dy,distance_px=math.hypot(dx,dy),squared_error=dx*dx+dy*dy)
        result.append(row)
    return result


def inner_mode_metrics(refined,centers):
    coarse=[centers[int(r['image_id'])] for r in refined]
    equal=blend_prediction_rows(coarse,refined,True)
    raw=raw_prediction_rows(refined)
    return dict(score=metrics_from_rows(equal)['rmse_coordinate_px'],
        refined_rmse=metrics_from_rows(refined)['rmse_coordinate_px'],
        raw_rmse=metrics_from_rows(raw)['rmse_coordinate_px'],
        mean_distance=metrics_from_rows(equal)['mean_distance_px'],
        fallbacks=sum(int(r['fallback']) for r in refined))


def phase_checkpoint(path,signature):
    state=torch.load(path,map_location='cpu',weights_only=True)
    if state.get('signature')!=signature:raise ValueError('Fine checkpoint signature differs.')
    return state


def train_diagnostic_fine(data,records,cache,config,folder,weights,evaluator=None,snapshots=()):
    validate_phase_request(config,evaluator is not None,snapshots)
    folder.mkdir(parents=True,exist_ok=True)
    signature=stable_hash(config)
    if (folder/'config.json').exists() and json.loads((folder/'config.json').read_text())!=config:
        raise ValueError('Fine phase configuration differs.')
    save_json(folder/'config.json',config)
    done_path=folder/'phase_complete.json'
    if done_path.exists():
        done=json.loads(done_path.read_text())
        if done['signature']!=signature:raise ValueError('Completed phase signature differs.')
        expected=([f'best_{mode}.pt' for mode in config['modes']] if evaluator else [f'epoch_{e}.pt' for e in snapshots])
        if any(not (folder/name).exists() for name in expected):raise ValueError('Full phase weights missing; do not restore from light ZIP.')
        return done
    seed_everything(config['seed']);torch.set_num_threads(2)
    dataset=DiagnosticPatchDataset(records,cache,config,True)
    loader=DataLoader(dataset,batch_size=8,shuffle=True,num_workers=0)
    last=folder/'last.pt'
    model=ResNet34CV(None if last.exists() else weights).to('cuda:0')
    backbone_ids={id(p) for p in model.backbone.parameters()}
    optimizer=torch.optim.AdamW([dict(params=model.backbone.parameters(),lr=5e-5),
        dict(params=[p for p in model.parameters() if id(p) not in backbone_ids],lr=5e-4)],weight_decay=1e-4)
    # Keep the60-epoch learning-rate policy identical for inner fitting and outer refit.
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,
        lr_lambda=lambda step:.05+.95*(1+math.cos(math.pi*step/config['schedule_epochs']))/2)
    scaler=torch.amp.GradScaler('cuda');history=[];start=0
    if last.exists():
        state=torch.load(last,map_location='cpu',weights_only=False)
        if state['signature']!=signature:raise ValueError('Optimizer signature differs.')
        model.load_state_dict(state['model'],strict=True);optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
        history,start=state['history'],int(state['epoch'])
        random.setstate(state['python_rng']);np.random.set_state(state['numpy_rng'])
        torch.set_rng_state(state['torch_rng']);torch.cuda.set_rng_state(state['cuda_rng'],device=0)
        del state
    best={mode:dict(score=float('inf'),epoch=0) for mode in config['modes']}
    if evaluator:
        for mode in best:
            if (folder/f'best_{mode}.pt').exists():
                saved=phase_checkpoint(folder/f'best_{mode}.pt',signature)
                best[mode]=dict(saved['selection']);del saved
    for epoch in range(start+1,config['epochs']+1):
        begin=time.perf_counter();model.train();total=count=skipped=0
        for images,points,_ in loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',enabled=True):
                logits=model(images.to('cuda:0'))
                if logits.shape!=(len(images),96,96):raise RuntimeError('Unexpected fine heatmap shape.')
                loss=heatmap_loss(logits,points.to('cuda:0'),1.5)
            if not torch.isfinite(loss):raise RuntimeError('Non-finite fine loss.')
            scaler.scale(loss).backward();scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.)
            old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()
            skipped+=int(scaler.get_scale()<old_scale);total+=float(loss.detach())*len(images);count+=len(images)
        if skipped==len(loader) and history and history[-1]['amp_skips']==len(loader):
            raise RuntimeError('Every optimizer step skipped in two consecutive epochs.')
        row=dict(epoch=epoch,train_ce=total/count,lr_backbone=optimizer.param_groups[0]['lr'],amp_skips=skipped)
        if evaluator:
            results=evaluator(model)
            for mode,values in results.items():
                row.update({f'{mode}_{k}':v for k,v in values.items()})
                if values['score']<best[mode]['score']:
                    best[mode]=dict(values,epoch=epoch)
                    atomic_torch_save(folder/f'best_{mode}.pt',dict(model=model.state_dict(),signature=signature,
                        epoch=epoch,selection=best[mode],config=config))
        elif epoch in snapshots:
            atomic_torch_save(folder/f'epoch_{epoch}.pt',dict(model=model.state_dict(),signature=signature,epoch=epoch,config=config))
        scheduler.step();row['seconds']=time.perf_counter()-begin;history.append(row)
        if epoch%5==0 or epoch==config['epochs']:
            atomic_torch_save(last,dict(model=model.state_dict(),optimizer=optimizer.state_dict(),
                scheduler=scheduler.state_dict(),scaler=scaler.state_dict(),signature=signature,epoch=epoch,history=history,
                python_rng=random.getstate(),numpy_rng=np.random.get_state(),torch_rng=torch.get_rng_state(),
                cuda_rng=torch.cuda.get_rng_state(device=0)))
        atomic_csv(folder/'history.csv',history)
        save_json(folder/'status.json',dict(state='training',epoch=epoch,signature=signature))
        scores=' '.join(f'{mode}={values["score"]:.2f}' for mode,values in best.items() if values['epoch'])
        print(f'Outer{config["outer_fold"]} {config["role"]}/{config["training_arm"]} {epoch:02}/{config["epochs"]}'
              f' CE={total/count:.4f} {scores} {row["seconds"]:.1f}s',flush=True)
    done=dict(signature=signature,best=best if evaluator else {},epochs=config['epochs'])
    save_json(done_path,done)
    if last.exists():last.unlink()
    del model,loader,dataset;gc.collect();torch.cuda.empty_cache()
    return done


def validate_inner_centers(rows,records):
    mapping=validate_test_rows(rows,records,-1)
    for r in records:
        p=mapping[int(r['image_id'])]
        if abs(float(p['gt_x'])-float(r['x']))>1e-6 or abs(float(p['gt_y'])-float(r['y']))>1e-6:
            raise ValueError('Inner center labels differ.')
    return mapping


def make_inner_coarse(data,fit,held,outer_ids,fold,folder,protocol,weights):
    folder.mkdir(parents=True,exist_ok=True)
    config=dict(model_name='resnet34',seed=31000+fold,epochs=protocol['inner_coarse_epochs'],schedule_epochs=80,
        image_size=512,batch_size=8,lr_backbone=1e-4,lr_decoder=1e-3,weight_decay=1e-4,sigma=2.4,
        weights='resnet34-b627a593.pth',pretrained=True,train_ids=[int(x) for x in fit.image_id],
        inner_validation_ids=[int(x) for x in held.image_id],forbidden_outer_ids=outer_ids,
        source_sha256=protocol['source_sha256'],coarse_fingerprint=protocol['coarse_fingerprint'])
    if set(config['train_ids'])&set(config['inner_validation_ids']) or (set(config['train_ids'])|set(config['inner_validation_ids']))&set(outer_ids):
        raise ValueError('Inner coarse image leakage.')
    signature=stable_hash(config);done_path=folder/'coarse_complete.json';pred_path=folder/'inner_validation.csv'
    if done_path.exists():
        done=json.loads(done_path.read_text())
        if done['signature']!=signature or not (folder/'checkpoints/best.pt').is_file():
            raise ValueError('Completed inner coarse output differs or lacks model.')
        return validate_inner_centers(read_rows(pred_path),held.to_dict('records'))
    last=folder/'checkpoints/last.pt'
    if (folder/'checkpoints/best.pt').exists() and not last.exists():
        raise ValueError('Unfinished inner coarse model lacks its optimizer checkpoint; restore full output.')
    if last.exists():
        state=torch.load(last,map_location='cpu',weights_only=False)
        if len(state['cuda_rng'])!=torch.cuda.device_count():
            raise ValueError('Resume auxiliary coarse training with the same visible GPU count.')
        del state
    # No held-out images or metrics are passed to the auxiliary coarse trainer.
    model,epoch=train_model(data,fit,None,folder/'checkpoints',config,torch.device('cuda:0'),
        model_builder=lambda cfg:ResNet34CV(weights),dataset_builder=FoveaDataset)
    if epoch!=protocol['inner_coarse_epochs']:raise ValueError('Inner coarse incomplete.')
    loader=DataLoader(FoveaDataset(data,held,512),batch_size=8,shuffle=False,num_workers=0)
    points=predict_cv(model,loader,torch.device('cuda:0'),5,False)
    rows=prediction_rows(held.to_dict('records'),points,-1,True)
    atomic_csv(pred_path,rows)
    save_json(done_path,dict(signature=signature,epoch=epoch,training_ids=config['train_ids'],
        validation_ids=config['inner_validation_ids'],validation_used_for_training=False,
        baseline_metrics=metrics_from_rows(rows)))
    last=folder/'checkpoints/last.pt'
    if last.exists():last.unlink()
    del model,loader;gc.collect();torch.cuda.empty_cache()
    return validate_inner_centers(rows,held.to_dict('records'))


def fine_config(protocol,fold,arm,role,train_ids,held_ids,forbidden_ids,epochs,targets=None):
    config=dict(version=DIAGNOSTIC_VERSION,source_sha256=protocol['source_sha256'],coarse_fingerprint=protocol['coarse_fingerprint'],
        outer_fold=fold,training_arm=arm,role=role,seed=(32000 if role=='inner' else 33000)+fold,
        epochs=epochs,schedule_epochs=protocol['inner_epochs'],image_size=384,crop_side=768,
        prior_sigma=protocol['prior_sigma'],train_ids=list(train_ids),evaluation_ids=list(held_ids),
        forbidden_ids=list(forbidden_ids),modes=['plain'] if arm=='wide' else ['plain','prior'],
        targets=targets or {})
    validate_phase_config(config)
    return config


def identity_rows(records,centers,fold,selection):
    rows=[]
    for r in records:
        c=centers[int(r['image_id'])];point=(float(c['Fovea_X']),float(c['Fovea_Y']))
        box=crop_geometry(point,768)
        row=make_refinement_row(r,point_to_patch(point,box),c,box,fold,256.)
        row.update(decoder='identity',refine_applied=0,selection_reason=selection['reason'])
        rows.append(row)
    return rows


def diagnostic_fold(prepared,fold,weights,cache):
    data,train,test,coarse,output,protocol=[prepared[k] for k in ('data','train','test','coarse','output','protocol')]
    development,fit,inner_val,outer=inner_partition(train,fold)
    job=output/f'outer_{fold}';job.mkdir(parents=True,exist_ok=True)
    outer_ids=[int(x) for x in outer.image_id]
    design=dict(protocol=protocol,fold=fold,fit_ids=[int(x) for x in fit.image_id],inner_val_ids=[int(x) for x in inner_val.image_id],outer_ids=outer_ids)
    signature=stable_hash(design)
    if (job/'fold_complete.json').exists():
        done=json.loads((job/'fold_complete.json').read_text())
        if done['signature']!=signature:raise ValueError('Completed outer fold differs.')
        if any(not (job/name).is_file() for name in done['required_models']):raise ValueError('Restore full output, not light results.')
        return
    split=[]
    for role,frame in [('inner_train',fit),('inner_validation',inner_val),('outer_validation',outer)]:
        split.extend(dict(image_id=int(r.image_id),pixel_sha256=r.pixel_sha256,role=role) for r in frame.itertuples())
    atomic_csv(job/'split_roles.csv',split)
    save_json(job/'design.json',design)
    centers=make_inner_coarse(data,fit,inner_val,outer_ids,fold,job/'inner_coarse',protocol,weights)
    baseline=metrics_from_rows(list(centers.values()))['rmse_coordinate_px']
    decisions={};inner_configs={}
    for arm in ('wide','small'):
        config=fine_config(protocol,fold,arm,'inner',[int(x) for x in fit.image_id],
            [int(x) for x in inner_val.image_id],outer_ids,protocol['inner_epochs'])
        inner_configs[arm]=config
        loader=DataLoader(DiagnosticPatchDataset(inner_val.to_dict('records'),cache,config,False,centers),batch_size=8,shuffle=False,num_workers=0)
        def evaluate(model):
            predictions=diagnostic_predict(model,loader,config,centers,-1,config['modes'])
            return {mode:inner_mode_metrics(rows,centers) for mode,rows in predictions.items()}
        done=train_diagnostic_fine(data,fit.to_dict('records'),cache,config,job/f'inner_{arm}',weights,evaluate)
        for mode,best in done['best'].items():
            name=arm if mode=='plain' else 'small_prior'
            decisions[name]=selected_inner_epoch(best,baseline,protocol['min_inner_gain'])
        del loader
    save_json(job/'inner_decisions.json',decisions)
    for arm in ('wide','small'):
        modes={'plain':arm} if arm=='wide' else {'plain':'small','prior':'small_prior'}
        enabled={mode:decisions[name]['epoch'] for mode,name in modes.items() if decisions[name]['enabled']}
        model=None;config=None
        if enabled:
            config=fine_config(protocol,fold,arm,'outer',[int(x) for x in development.image_id],[],outer_ids,max(enabled.values()),enabled)
            phase=job/f'outer_{arm}'
            train_diagnostic_fine(data,development.to_dict('records'),cache,config,phase,weights,snapshots=sorted(set(enabled.values())))
            model=ResNet34CV().to('cuda:0')
        for mode,name in modes.items():
            decision=decisions[name]
            for subset,frame,prior in [('validation',outer,coarse['validation'][fold]),('test',test,coarse['test'][fold])]:
                records=frame.to_dict('records')
                if decision['enabled']:
                    state=phase_checkpoint(job/f'outer_{arm}/epoch_{decision["epoch"]}.pt',stable_hash(config))
                    model.load_state_dict(state['model'],strict=True);del state
                    loader=DataLoader(DiagnosticPatchDataset(records,cache,config,False,prior),batch_size=8,shuffle=False,num_workers=0)
                    rows=diagnostic_predict(model,loader,config,prior,fold,[mode])[mode]
                    for row in rows:row['selection_reason']=decision['reason']
                    del loader
                else:rows=identity_rows(records,prior,fold,decision)
                atomic_csv(job/f'{subset}_{name}.csv',rows)
        if model is not None:del model
        gc.collect();torch.cuda.empty_cache()
    required=[p.relative_to(job).as_posix() for p in job.rglob('*.pt')]
    save_json(job/'fold_complete.json',dict(signature=signature,decisions=decisions,required_models=required,
        outer_labels_used_for_epoch_selection=False,predictions_complete=True))


def diagnostic_reports(prepared):
    data,train,test,coarse,output=[prepared[k] for k in ('data','train','test','coarse','output')]
    available=[f for f in range(5) if (output/f'outer_{f}/fold_complete.json').exists()]
    selection=dict(complete=len(available)==5,completed_folds=available,candidate_rule='Predeclared small-prior equal blend; per-fold inner5% gain gate',
        selected_by_outer_scores=False,official_metric_confirmed=False,agent_submitted_to_competition=False,
        limitation='Outer fine labels never choose epochs/gates. Legacy coarse checkpoints already used these outer labels for stopping; comparison is conditional development evaluation, not unbiased validation of the entire cascade.')
    selection['smoke_test']=bool(prepared['protocol']['smoke_test'])
    if len(available)!=5 or selection['smoke_test']:
        selection['complete']=False
        save_json(output/'selection.json',selection);return False
    records,test_records=train.to_dict('records'),test.to_dict('records')
    coarse_oof=validate_oof([r for f in range(5) for r in coarse['validation'][f].values()],records)
    atomic_csv(output/'oof_coarse512.csv',coarse_oof)
    baseline_test=average_test_folds({f:list(coarse['test'][f].values()) for f in range(5)},test_records)
    atomic_csv(output/'test_coarse512.csv',baseline_test)
    write_submission(data/'sample_submission.csv',baseline_test,output/'submission_coarse512.csv')
    comparisons=[dict(arm='coarse512',**metrics_from_rows(coarse_oof))];decision_rows=[]
    for variant in VARIANTS:
        refined=[];test_folds={};active=0
        for f in range(5):
            job=output/f'outer_{f}'
            done=json.loads((job/'fold_complete.json').read_text())
            development,fit,inner_val,outer=inner_partition(train,f)
            design=dict(protocol=prepared['protocol'],fold=f,fit_ids=[int(x) for x in fit.image_id],
                inner_val_ids=[int(x) for x in inner_val.image_id],outer_ids=[int(x) for x in outer.image_id])
            if done['signature']!=stable_hash(design) or not done['predictions_complete'] or done['outer_labels_used_for_epoch_selection']:
                raise ValueError('Outer completion provenance differs.')
            decisions=done['decisions'][variant]
            recomputed=selected_inner_epoch(dict(score=decisions['inner_score'],epoch=decisions['inner_best_epoch']),
                decisions['inner_baseline'],prepared['protocol']['min_inner_gain'])
            if decisions!=recomputed:raise ValueError('Inner gate/epoch rule differs from predeclared selection.')
            decision_rows.append(dict(variant=variant,outer_fold=f,**decisions))
            active+=int(decisions['enabled'])
            held=train[train.fold==f].to_dict('records')
            rows=validate_oof(read_rows(job/f'validation_{variant}.csv'),held)
            validate_test_rows(rows,held,f);refined.extend(rows)
            test_folds[f]=read_rows(job/f'test_{variant}.csv')
            validate_test_rows(test_folds[f],test_records,f)
        refined=validate_oof(refined,records)
        raw=raw_prediction_rows(refined)
        for suffix,rows,preds in [('refined',refined,test_folds),
            ('equal',blend_prediction_rows(coarse_oof,refined,True),
             {f:blend_prediction_rows(list(coarse['test'][f].values()),test_folds[f],False) for f in range(5)})]:
            name=f'{variant}_{suffix}'
            atomic_csv(output/f'oof_{name}.csv',rows)
            mean=average_test_folds(preds,test_records)
            atomic_csv(output/f'test_{name}.csv',mean)
            write_submission(data/'sample_submission.csv',mean,output/f'submission_{name}.csv')
            comparisons.append(dict(arm=name,active_folds=active,**metrics_from_rows(rows)))
        atomic_csv(output/f'oof_{variant}_raw.csv',raw)
        comparisons.append(dict(arm=f'{variant}_raw_diagnostic',active_folds=active,**metrics_from_rows(raw)))
    atomic_csv(output/'comparison.csv',comparisons,fields=['arm','active_folds','n','mean_distance_px','median_distance_px',
        'p90_distance_px','max_distance_px','mae_x_px','mae_y_px','rmse_coordinate_px'])
    atomic_csv(output/'inner_decisions.csv',decision_rows)
    candidate=output/'submission_small_prior_equal.csv'
    shutil.copy2(candidate,output/'submission_refine_v3_candidate.csv')
    selection.update(candidate='submission_refine_v3_candidate.csv',public_coarse_reference=14.16538,
        public_best_reference=12.31624,candidate_public_score=None)
    save_json(output/'selection.json',selection)
    print(pd.DataFrame(comparisons).to_string(index=False),flush=True)
    return True


def diagnostic_archive(output,full=False):
    target=output.parent/'task3_refine_v3_full_backup.zip' if full else output/'task3_refine_v3_results_light.zip'
    temporary=target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as archive:
        for path in sorted(output.rglob('*')):
            if not path.is_file() or path.is_symlink() or path.suffix in ('.tmp','.zip'):continue
            if full or path.suffix in ('.csv','.json','.png','.md'):archive.write(path,path.relative_to(output))
    temporary.replace(target);return target


def prepare_diagnostic(coarse_source,data_dir=None,output_dir='/kaggle/working/task3_refine_diagnostic_v3',resume_dir=None,smoke=False):
    if not Path('/kaggle/working').is_dir() or not torch.cuda.is_available():
        raise RuntimeError('Use Kaggle GPU; no local/CPU training was started.')
    data,output=find_data(data_dir),Path(output_dir).resolve()
    manifest=load_manifest(data);train=assign_folds(manifest[manifest.subset=='train'],42,5);test=manifest[manifest.subset=='test'].copy()
    if len(train)!=80 or len(test)!=20:raise ValueError('Unexpected image counts.')
    coarse=validate_coarse_bundle(coarse_source,train.to_dict('records'),test.to_dict('records'))
    protocol=dict(version=DIAGNOSTIC_VERSION,source_sha256=diagnostic_sources(),coarse_fingerprint=coarse['fingerprint'],
        inner_coarse_epochs=2 if smoke else 43,inner_epochs=2 if smoke else 60,prior_sigma=48.,min_inner_gain=.05,
        inner_split_seed_base=7000,training_arms=['wide','small'],candidate_variant='small_prior_equal',
        outer_epochs_selected_by_inner=True,legacy_coarse_outer_selection_bias_remains=True,smoke_test=bool(smoke))
    output.mkdir(parents=True,exist_ok=True)
    if shutil.disk_usage(output).free<9*1024**3:raise RuntimeError('Need9 GiB free for nested phases, atomic saves and full backup.')
    restore_backup(resume_dir,output,protocol)
    if (output/'protocol.json').exists() and json.loads((output/'protocol.json').read_text())!=protocol:
        raise ValueError('Existing diagnostic protocol differs; use a new output directory.')
    save_json(output/'protocol.json',protocol);train.to_csv(output/'split.csv',index=False);manifest.to_csv(output/'manifest.csv',index=False)
    save_json(output/'coarse_source.json',dict(fingerprint=coarse['fingerprint'],legacy_protocol=coarse['protocol']))
    save_json(output/'environment.json',dict(torch=torch.__version__,numpy=np.__version__,gpu=torch.cuda.get_device_name(0),
        visible_gpu_count=torch.cuda.device_count(),device_policy='cuda:0 sequential phases; original image RGB cache up to about2 GiB'))
    print('Fixed coarse512 OOF:',coarse['baseline_metrics'],flush=True)
    print('Auxiliary inner coarse + wide/small inner fits + at most2 outer refits per fold; up to25 training phases.',flush=True)
    return dict(data=data,output=output,train=train,test=test,coarse=coarse,protocol=protocol)


def run_diagnostic(prepared,run_folds=(0,1,2,3,4),weights=None,full_backup=True):
    if not Path('/kaggle/working').is_dir() or not torch.cuda.is_available():raise RuntimeError('Kaggle GPU required.')
    if not run_folds or len(set(run_folds))!=len(run_folds) or not set(run_folds)<=set(range(5)):
        raise ValueError('Invalid outer fold schedule.')
    output=prepared['output'];cache=OriginalImageCache(prepared['data'])
    pending=any(not (output/f'outer_{f}/fold_complete.json').exists() for f in run_folds)
    weights=resolve_weights(output,weights) if pending else None
    with training_lock(output):
        try:
            for fold in run_folds:
                save_json(output/'status.json',dict(state='running',outer_fold=fold,updated_unix=time.time()))
                diagnostic_fold(prepared,fold,weights,cache)
                diagnostic_reports(prepared);diagnostic_archive(output)
            complete=diagnostic_reports(prepared)
            save_json(output/'status.json',dict(state='completed' if complete else 'partial_completed',updated_unix=time.time()))
            diagnostic_archive(output)
            if complete and full_backup:
                backup=diagnostic_archive(output,True)
                with zipfile.ZipFile(backup) as archive:
                    for f in range(5):
                        done=json.loads((output/f'outer_{f}/fold_complete.json').read_text())
                        if any(f'outer_{f}/{name}' not in archive.namelist() for name in done['required_models']):
                            raise RuntimeError('Full backup lacks a required model.')
                print('Full backup:',backup,flush=True)
        except BaseException as error:
            save_json(output/'status.json',dict(state='interrupted' if isinstance(error,KeyboardInterrupt) else 'failed',error=str(error),updated_unix=time.time()))
            diagnostic_archive(output);raise
    return output


## 2. 预检查

核对80/20图像、固定分组、旧粗预测和代码指纹。内层划分按像素组进行，重复图不会跨内层或外层边界。每个辅助粗模型固定43轮、512输入、80轮余弦周期；没有把内层验证图传给其训练函数。

fine inner在约51张上训练60轮，内层13张只用于选轮次。wide与small使用相同初始化/批次种子，中心采样消耗相同随机数数量，其余增强一致。small中心扰动为90%截断高斯σ16（每轴限制±48原图像素），10%均匀±64；不是从外层验证误差或公开榜搜索参数。

small_prior对同一网络热图加粗点中心的高斯log先验，σ48原图像素，只影响解码，不增加训练输入或额外训练一个网络。它是位置先验，不是全局特征融合，也不是ZIAN复现。


In [ ]:
PREPARED=prepare_diagnostic(COARSE_SOURCE,data_dir=DATA_DIR,output_dir=OUTPUT_DIR,
    resume_dir=RESUME_DIR,smoke=SMOKE_TEST)
print('GPU:',torch.cuda.get_device_name(0))
print('Output:',OUTPUT_DIR)
for fold in RUN_FOLDS:
    dev,fit,inner_val,outer=inner_partition(PREPARED['train'],fold)
    print(f'Outer{fold}: inner train={len(fit)}, inner validation={len(inner_val)}, final refit={len(dev)}, outer evaluation={len(outer)}')


## 3. 运行受控诊断

内层按预先固定的“粗点与精修点各50%”系统的坐标RMSE选轮次。每个对照只有达到相对内层粗基线至少5%改善，才在该外层折启用精修；否则这折直接使用粗预测。5%门槛、48像素先验、扰动分布和256像素回退都预先固定，不用外层结果调节。

选好轮次后从ImageNet重新初始化，使用外层全部64图训练对应轮数，学习率仍沿用60轮周期。small和small_prior若选中不同轮次，只训练一次small到较大轮次，保存两份指定快照。外层标签只在最终报告里评分。

原始精修、回退后精修、50/50系统三套指标都会保存。只要网络仍跳点，即使回退保护了总分，也会在raw_diagnostic指标和逐图CSV里显示。


In [ ]:
RESULT_DIR=run_diagnostic(PREPARED,run_folds=RUN_FOLDS,weights=WEIGHTS,
    full_backup=CREATE_FULL_BACKUP)


## 4. 查看结果

comparison.csv按固定顺序列出粗基线及各组原始/回退/等权指标，不根据外层排名自动挑选方案。预先指定候选为small_prior_equal，是否启用精修由各折内层门控决定；可能所有折都回到粗点。先检查结果再考虑比赛提交，不要因为生成文件就认为一定改进。

全套五折完成且非smoke时才导出submission_refine_v3_candidate.csv。此Notebook不提交比赛，也不会覆盖原12.31624版本。


In [ ]:
from IPython.display import display
print((RESULT_DIR/'selection.json').read_text(encoding='utf-8'))
if (RESULT_DIR/'comparison.csv').exists():
    display(pd.read_csv(RESULT_DIR/'comparison.csv'))
    display(pd.read_csv(RESULT_DIR/'inner_decisions.csv'))
candidate=RESULT_DIR/'submission_refine_v3_candidate.csv'
if candidate.exists():
    display(pd.read_csv(candidate))
    print('Predeclared candidate:',candidate)
print('Light ZIP:',RESULT_DIR/'task3_refine_v3_results_light.zip')
print('Full backup:',RESULT_DIR.parent/'task3_refine_v3_full_backup.zip')


## 5. 保存与恢复

完成后交回task3_refine_v3_results_light.zip分析。完整备份task3_refine_v3_full_backup.zip自动保存所有辅助/精修权重和结果，体积会比上一版大，建议下载到D盘。模型份数随内层门控和所选轮次变化，不能用固定数量判断是否完整；归档会检查每折元数据列出的全部required_models。

每5轮保存精修optimizer断点；辅助粗模型沿用逐轮保存。完成一个阶段后删除其自身last.pt，保留选择所需权重或外层指定轮次快照。未完成阶段恢复需本版完整Output，不能只用轻量ZIP。

在新会话挂载完整Output解压目录，设置RESUME_DIR为直接含本版protocol.json的目录；保持同一Notebook、配置与比赛图片。辅助粗模型optimizer恢复需相同GPU数量。内置旧粗预测会自动恢复，不需要再次上传。

本地准备不导入Torch、不训练模型。低内存检查覆盖数据边界、配对采样、先验坐标、内层门控、快照、导出与归档协议；真实GPU前向/反向和断点恢复仍需Kaggle执行。
